In [ ]:
%load_ext autoreload
%autoreload 2

# Enumerator

The `Enumerator` served tool generates analogue libraries from a single parent `Ligand`. It supports four `job_type` values:

- `SCAFFOLD` — MMP: grow a fragment at one attachment atom
- `ANALOGUE` — MMP: swap a connected fragment
- `AVAILABLE_REACTIONS` — discover named-reaction sites on the parent (no CSV)
- `REACTION` — enumerate products against the Enamine fragment library

Every mode is configured in `__init__` and executed with a blocking `run()` that returns a `pandas.DataFrame`.

In [ ]:
from deeporigin.drug_discovery import Enumerator, Ligand
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient()
client

## Create a parent ligand

The enumerator takes a single parent `Ligand`. Its SMILES is sent inline; its optional `id` is echoed back as `parent_ligand_id` in the results. Here we use bromobenzene.

In [ ]:
parent = Ligand.from_smiles("Brc1ccccc1")
parent

## MMP mode 1: SCAFFOLD

`SCAFFOLD` grows a new fragment at a single attachment atom. Pass the RDKit atom index as `replace_ix`. The result is a descriptor-enriched DataFrame.

In [ ]:
scaffold = Enumerator(ligand=parent, job_type="SCAFFOLD", replace_ix=3)
df = scaffold.run()
df.head()

In [ ]:
# Whether the run hit the platform enumeration cap
scaffold.cap_hit

## MMP mode 2: ANALOGUE

`ANALOGUE` swaps out a connected fragment defined by one or more atom indices. Tune the CReM search with `radius` (1-5) and `max_fragment_size` (1-15).

In [ ]:
analogue = Enumerator(
    ligand=parent,
    job_type="ANALOGUE",
    replace_ix=[3, 4],
    radius=2,
    max_fragment_size=8,
)
analogue.run().head()

## Discovery: AVAILABLE_REACTIONS

Choosing valid reaction sites by hand is error-prone, so run `AVAILABLE_REACTIONS` first. It returns one row per matched named-reaction site, with the `atom_indices` you feed into a `REACTION` run. This mode writes no CSV — the DataFrame is built from the inline result list.

In [ ]:
sites = Enumerator(ligand=parent, job_type="AVAILABLE_REACTIONS").run()
sites

## Enumeration: REACTION

Pick the rows you want from the discovery table and pass them verbatim as `reaction_sites`. Each site must match a hit from `AVAILABLE_REACTIONS` on the same parent.

In [ ]:
suzuki_rows = sites[sites["reaction_id"] == "suzuki"]
reaction_sites = [
    {
        "reaction_id": row["reaction_id"],
        "reactant_role": row["reactant_role"],
        "atom_indices": list(row["atom_indices"]),
    }
    for _, row in suzuki_rows.iterrows()
]
reaction_sites

In [ ]:
reaction = Enumerator(
    ligand=parent,
    job_type="REACTION",
    reaction_sites=reaction_sites,
)
df = reaction.run()
df.head()

## Reload an existing run

Reconstruct an `Enumerator` from a tools execution ID and re-fetch its results without re-running.

In [ ]:
reloaded = Enumerator.from_id(reaction.id)
reloaded.get_results().head()